<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 1's own methods explicitly license a route into the standard types: *"simple
regressions or classification, if you define a target"* and *"feature importance from a simple
model."* That's the route I'm using: define a binary target, fit a simple classifier, and read its
feature importances as a rigorous, multivariate version of the grouped effect-size work from w01, an instrument for measuring signals, not a deployed decision system.

So concretely: **classification**, where the model's job is to answer "do these signals, combined,
carry more information than any one of them alone?".

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    import subprocess
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started (works from notebooks/ or work/notebooks/)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head())


             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining` = `trend_direction == "down"`**, the same starter-dataset proxy the
pipeline already ships as `is_declining_label`.

This is an available-now proxy, not the outcome I'd ultimately want. It's a bucket computed from
the *current* 90-day window, not a future
observed outcome, exactly the label-trap the flyrank-data skill and lane guide both call out. I'm
using it here because this notebook is about proving the *task framing* works at all, cheaply,
before investing in a real forward-looking label. Planned upgrade, once I'm past the provisional-lane
window: a warehouse-built label like "prior 90 days of features → next 30 days decline".

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"is_declining proxy rate: {df['is_declining'].mean()*100:.1f}% of {len(df):,} starter rows")
print("trend_direction (and is_declining) is computed from the CURRENT 90-day window, not a future outcome")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining proxy rate: 54.2% of 30,000 starter rows
trend_direction (and is_declining) is computed from the CURRENT 90-day window, not a future outcome


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: ROC curve,  area under curve on a client-held-out test set.** Also, considering precision recall, and F1 score. The choice is appropriate in this case as the data is fairly balanced (the starter one is, I expect the original one also to be). Along them, I also consider analyzing the confusion matrix.

I read AUC next to the test set's base rate, so it's judged
against a coin flip, not in a vacuum. Feature importance ranking is also important: we have to answer which one of them have (the most) impact on visibility, clicks, engagement, etc.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

base_rate = df["is_declining"].mean()
print(f"Base rate for is_declining across the full starter slice: {base_rate*100:.1f}%")
print("-> a model needs to clear this (and AUC 0.50, a coin flip) to count as real signal")


Base rate for is_declining across the full starter slice: 54.2%
-> a model needs to clear this (and AUC 0.50, a coin flip) to count as real signal


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (one page)**, summarized over its trailing 90-day window

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_cols = ["content_id", "client_id", "content_type", "impressions_90d", "avg_position",
             "ctr", "days_since_last_update", "trend_direction", "is_declining"]

df[unit_cols].head(5)


,content_id,client_id,content_type,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,0.76,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,0.05,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,0.09,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,0.49,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,0.13,14,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

First of all, there are many features, and it is unclear which ones are important. We can observe that the random forest model performed near the baseline and we did not observe any evident best predictors, and the effect can be seen when considering them together. The latter means it would be hard to extract 1-2 predictors and create an if-statement

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

df["has_keyword_data"] = df["search_volume"].notna().astype(int)  # feedly articles: keyword data ~100% missing (gotcha)
df["has_word_count"] = df["word_count"].notna().astype(int)       # keyword articles: ~28% missing word_count (gotcha)

num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
            "content_age_days",
            "days_since_last_update", "ctr"]
flag_cols = ["has_keyword_data", "has_word_count"]

X = df[num_cols].fillna(0).copy()   # safe: every fillna is paired with an explicit has_-flag above
for f in flag_cols:
    X[f] = df[f]
y = df["is_declining"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(df["client_id"].iloc[train_idx]) & set(df["client_id"].iloc[test_idx])
print(f"Client-grouped split: {df['client_id'].iloc[train_idx].nunique()} train clients, "
      f"{df['client_id'].iloc[test_idx].nunique()} test clients, {len(overlap)} overlapping.")
print()

clf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                              random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
combined_auc = roc_auc_score(y_test, proba)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)


print(f"\nCombined-signal model AUC (Random Forest, all {X.shape[1]} features): {combined_auc:.3f}")
print(f"Random Forest Accuracy: {accuracy * 100:.2f}%")
print(f"Test-set base rate: {y_test.mean()*100:.1f}%")

importances = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop signals by feature importance:")
print(importances.head(6).round(3).to_string())


Client-grouped split: 24 train clients, 8 test clients, 0 overlapping.


Combined-signal model AUC (Random Forest, all 10 features): 0.519
Random Forest Accuracy: 52.21%
Test-set base rate: 51.7%

Top signals by feature importance:
content_age_days          0.295
days_since_last_update    0.139
word_count                0.138
has_keyword_data          0.137
char_count                0.123
ctr                       0.096


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.